# Causals Graph: first 10 minutes

This notebook uses a **62-node, 9-edge teaching sample** from the fixed `v0.1.0-2026-10-09` release. It runs with Python's standard library; no NewsGraph code, database, API key, or GPU is needed. The sample is deliberately selected, **not representative**, and its recorded links are **not independently verified causal effects**. See the [data card](https://huggingface.co/datasets/causals-lab/causals-graph/blob/main/v0.1.0-2026-10-09/DATA_CARD.md) before using the full data.

## 1. Locate the sample

When run from a clone of the community repository, the files are already present. A hosted notebook downloads only the three small sample files from the public repository, with a 15-second timeout per request.

In [ ]:
from pathlib import Path
import hashlib, json, urllib.request
from collections import Counter

sample = next((p for p in (Path('sample'), Path('quickstart/sample')) if (p / 'manifest.json').exists()), None)
if sample is None:
    sample = Path('sample')
    sample.mkdir(exist_ok=True)
    base_url = 'https://raw.githubusercontent.com/causals-lab/community/main/quickstart/sample/'
    for filename in ('manifest.json', 'nodes.jsonl', 'edges.jsonl'):
        with urllib.request.urlopen(base_url + filename, timeout=15) as response:
            (sample / filename).write_bytes(response.read())
print('Sample directory:', sample.resolve())

## 2. Check the files, then join nodes and edges

An edge's `source_id` and `target_id` refer to node `id`. Its `source_episode_ids` refer to source-news nodes—not to article text or URLs.

In [ ]:
manifest = json.loads((sample / 'manifest.json').read_text(encoding='utf-8'))
for entry in manifest['files']:
    path = sample / entry['file']
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    assert digest == entry['sha256'], f'Checksum mismatch: {path}'

def read_jsonl(path):
    with path.open(encoding='utf-8') as stream:
        for line in stream:
            yield json.loads(line)

nodes = {node['id']: node for node in read_jsonl(sample / 'nodes.jsonl')}
edges = list(read_jsonl(sample / 'edges.jsonl'))
assert len(nodes) == next(f['records'] for f in manifest['files'] if f['file'] == 'nodes.jsonl')
assert len(edges) == next(f['records'] for f in manifest['files'] if f['file'] == 'edges.jsonl')
assert all(e['source_id'] in nodes and e['target_id'] in nodes for e in edges)
assert all(source in nodes for e in edges for source in e.get('source_episode_ids', []))
print(f'{len(nodes)} nodes, {len(edges)} edges')
print('Relations:', dict(Counter(e['relation'] for e in edges)))

## 3. Inspect three neighborhoods

These are stored relations. The arrow below is the stored edge direction; it is **not necessarily the direction of causal propagation**.

In [ ]:
def label(node_id):
    return nodes[node_id].get('name') or f'[unnamed {node_id[:8]}]'

for center in ('Red Sea', 'Federal Reserve', 'Wheat'):
    center_ids = {node_id for node_id, node in nodes.items() if node.get('name') == center}
    print(f'\n{center}:')
    for edge in edges:
        if edge['source_id'] in center_ids or edge['target_id'] in center_ids:
            print(f"  {label(edge['source_id'])} --{edge['relation']}--> {label(edge['target_id'])}")

## 4. Inspect provenance without overstating it

A source-news ID is a link inside this released graph, not an independently checked citation. The original article URL and text are not included in v0.1.0.

In [ ]:
edge = next(e for e in edges if label(e['source_id']) == 'Houthi militia' and label(e['target_id']) == 'Red Sea')
print('Edge ID:', edge['id'])
print('Stored relation:', edge['relation'])
print('Source-news IDs:', len(edge.get('source_episode_ids', [])))
print('First source-news node:', nodes[edge['source_episode_ids'][0]])
print('This record is not a measured causal effect.')

## Continue with the full release

Download the fixed snapshot from [causals.org/download](https://causals.org/download/) or [Hugging Face](https://huggingface.co/datasets/causals-lab/causals-graph). The full files are gzip-compressed JSON Lines; stream them with `gzip.open(path, 'rt', encoding='utf-8')` instead of loading every edge into a notebook cell. Consult the data card for missing names, source limitations, temporal fields, relation semantics, and license terms. To report a problem with a record, use [Contribute](https://causals.org/contribute.html); send personal or removal concerns to `data@causals.org`.